In [ ]:
# ===================== CELL 1: build HOST-focused scene + store what will be shown to LLM =====================
import json, time
from datetime import datetime
import numpy as np
import pandas as pd

HOST_CSV = "host_stats_onos.csv"          # <-- your host stats CSV
HOST_META = "host_metadata.json"    # <-- optional (mac->host mapping)
WINDOW_MINUTES = 5
TREND_POINTS = 10
SCENE_LOG = "llm_scene_log.jsonl"
DECISION_LOG = "llm_decision_log.jsonl"

def now_iso(): 
    return datetime.now().isoformat()

def build_host_stats(df, trend_points=10):
    """
    Input: df already filtered to the desired time window
    Output: list of host objects (one per MAC)
    """
    hosts = []
    for mac, g in df.groupby("host_mac"):
        g = g.sort_values("timestamp").tail(trend_points)

        tx_pps = g["tx_pps"].astype(float).values
        rx_pps = g["rx_pps"].astype(float).values

        tx_kbps = g["tx_mbps"].astype(float).values * 1000.0
        rx_kbps = g["rx_mbps"].astype(float).values * 1000.0

        if len(tx_pps) < 2:
            continue

        host_obj = {
            "mac": str(mac),

            "tx_pps_trend": [round(x, 2) for x in tx_pps.tolist()],
            "rx_pps_trend": [round(x, 2) for x in rx_pps.tolist()],

            "tx_kbps_trend": [round(x, 2) for x in tx_kbps.tolist()],
            "rx_kbps_trend": [round(x, 2) for x in rx_kbps.tolist()],

            "tx_pps_mean": round(float(np.mean(tx_pps)), 2),
            "rx_pps_mean": round(float(np.mean(rx_pps)), 2),

            "tx_pps_std": round(float(np.std(tx_pps)), 3),
            "rx_pps_std": round(float(np.std(rx_pps)), 3),

            "tx_pps_max": round(float(np.max(tx_pps)), 2),
            "rx_pps_max": round(float(np.max(rx_pps)), 2),

            "tx_kbps_mean": round(float(np.mean(tx_kbps)), 2),
            "rx_kbps_mean": round(float(np.mean(rx_kbps)), 2),

            "tx_kbps_std": round(float(np.std(tx_kbps)), 3),
            "rx_kbps_std": round(float(np.std(rx_kbps)), 3),

            "tx_kbps_max": round(float(np.max(tx_kbps)), 2),
            "rx_kbps_max": round(float(np.max(rx_kbps)), 2),

            "tx_pps_delta": round(float(tx_pps[-1] - tx_pps[-2]), 2),
            "rx_pps_delta": round(float(rx_pps[-1] - rx_pps[-2]), 2),

            "tx_kbps_delta": round(float(tx_kbps[-1] - tx_kbps[-2]), 2),
            "rx_kbps_delta": round(float(rx_kbps[-1] - rx_kbps[-2]), 2),
        }

        hosts.append(host_obj)

    return hosts

def build_scene_and_prompt(model_name="llama3", last_k_decisions=10):
    # ---- read host stats ----
    df = pd.read_csv(HOST_CSV)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    if df.empty:
        return None, None

    # ---- filter last WINDOW_MINUTES ----
    t_end = df["timestamp"].max()
    t_min = t_end - pd.Timedelta(minutes=WINDOW_MINUTES)
    dfw = df[df["timestamp"] >= t_min].copy()
    if dfw.empty:
        return None, None
    
    # ---- metadata ----
    try:
        with open(HOST_META, "r", encoding="utf-8") as f:
            host_meta_obj = json.load(f)
    except FileNotFoundError:
        host_meta_obj = {"note": f"{HOST_META} not found"}

    # ---- build host stats via separate function ----
    hosts = build_host_stats(dfw, trend_points=TREND_POINTS)

    #///// filter //////
    hosts = [
        {
            "mac": h["mac"],
            "tx_pps_trend": h["tx_pps_trend"],
            "rx_pps_trend": h["rx_pps_trend"],
            "tx_kbps_trend": h["tx_kbps_trend"],
            "rx_kbps_trend": h["rx_kbps_trend"]
        }
        for h in hosts
    ] 

    # ---- last K decisions ----
    last_decisions = []
    try:
        with open(DECISION_LOG, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    last_decisions.append(json.loads(line))
        last_decisions = last_decisions[-last_k_decisions:]
    except FileNotFoundError:
        last_decisions = []

    # ---- HOST-level scene (what we show to LLM) ----
    scene = {
        "ts": now_iso(),
        "window_minutes": WINDOW_MINUTES,
        "trend_points": TREND_POINTS,
        "window_end_time": str(t_end),
        "counts": {
            "unique_host_macs": int(len(hosts)),
            "raw_rows_in_window": int(len(df)),
        },
        "host_metadata": host_meta_obj,
        "host_stats": hosts,
        "last_decisions": last_decisions
    }
    prompt = f"""
You are an SDN network anomaly detection function. You need the decide on the below task.

Task:
Select host MAC addresses that show increasing traffic.

Indicators of abnormal behavior:
- rx_pps_std > 0
- tx_pps_std unusually high
- tx_pps_delta large

If no host meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {{"mac":"MAC_ADDRESS","reason":"short_reason"}}
 ]
}}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{{' and end with '}}'.

You have to observe the host stats and outut the finding. You must follow the Rules and must respond in json file.
host_stats:
{json.dumps(hosts)}

Return JSON.
""".strip()

    # ---- log exactly what LLM saw ----
    with open(SCENE_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "ts": now_iso(),
            "model": model_name,
            "scene": scene,
            "prompt": prompt
        }) + "\n")

    print(prompt)
    return scene, prompt

In [ ]:
# ===================== CELL 2: call LLM (Ollama Cloud REST API) + store output =====================
import json
import requests
import time
import os

# ---- Cloud endpoint + auth ----
CLOUD_URL = "https://ollama.com/api/chat"
API_KEY = read_api_key()

MODEL_NAME = "gpt-oss:20b-cloud"

scene, prompt = build_scene_and_prompt(
    model_name=MODEL_NAME,
    last_k_decisions=10
)

if prompt is None:
    print("[!] No data yet (empty/missing CSV rows).")
else:
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an SDN observer and you will detect data anomaly.\n"
                    "Return ONLY valid JSON. No prose."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        # optional knobs (cloud supports options like local)
        "options": {
            "temperature": 0,
            "top_p": 0.9
        }
    }

    headers = {
        "Authorization": f"Bearer {API_KEY.strip()}",
        "Content-Type": "application/json"
    }

    start_time = time.time()
    r = requests.post(CLOUD_URL, headers=headers, json=payload, timeout=360)

    print("Status:", r.status_code)
    if r.status_code != 200:
        print("Error body:", r.text[:500])

    r.raise_for_status()
    latency = time.time() - start_time

    response_json = r.json()

    # Ollama /api/chat returns: {"message": {"role": "...", "content": "..."} , ...}
    out = response_json.get("message", {}).get("content", "").strip()

    print("\n\n===== MODEL OUTPUT =====\n")
    print(out)
    print(f"\n[Latency: {latency:.2f} sec]")

    with open(DECISION_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps({
            "ts": now_iso(),
            "model": MODEL_NAME,
            "latency_sec": latency,
            "llm_raw": out,
            "scene_ts": scene.get("ts") if scene else None
        }) + "\n")

You are an SDN network anomaly detection function. You need the decide on the below task.

Task:
Select host MAC addresses that show increasing traffic.

Indicators of abnormal behavior:
- rx_pps_std > 0
- tx_pps_std unusually high
- tx_pps_delta large

If no host meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {"mac":"MAC_ADDRESS","reason":"short_reason"}
 ]
}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{' and end with '}'.

You have to observe the host stats and outut the finding. You must follow the Rules and must respond in json file.
host_stats:
[{"mac": "00:00:00:00:00:01", "tx_pps_trend": [0.67, 0.67, 0.6, 0.66, 0.67, 0.6, 0.67, 0.67, 0.6, 0.67], "rx_pps_trend": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "tx_kbps_trend": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,